# 🎬 Yotuubef Studio — Web GUI for Google Colab

Welcome to **Yotuubef Studio**! This notebook launches the complete interactive Web Graphical User Interface (GUI) directly inside your Google Colab session.

### 🚀 Quick Start Instructions:
1. **Set GPU Runtime**: Go to **Runtime -> Change runtime type** and select **GPU** (recommended for video rendering & voiceover TTS).
2. **Set Colab Secrets**: Click the key icon (🔑) on the left sidebar and add your API keys:
   - `REDDIT_CLIENT_ID`
   - `REDDIT_CLIENT_SECRET`
   - `NVIDIA_NIM_API_KEY`
   - `HACKCLUB_SEARCH_API_KEY` (optional)
   - `YOUTUBE_TOKEN_JSON` (optional)
3. **Run Cells 1 to 4 in order**.
4. Click the **👉 OPEN YOTUUBEF STUDIO WORKSPACE 🚀** button in Cell 4 to open the full Web Studio!

In [ ]:
# Cell 1: Environment Setup & Dependencies
import os
import shlex
import subprocess
from pathlib import Path

def run(cmd: str, allow_fail: bool = False) -> None:
    print(f"$ {cmd}")
    completed = subprocess.run(cmd, shell=True, text=True)
    if completed.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")

print("📦 Installing system dependencies...")
run("apt-get -qq update")
run("apt-get -qq install -y ffmpeg sox git gcc g++ nodejs npm")

REPO_URL = "https://github.com/beenycool/yotuubef.git"
REPO_DIR = Path("/content/yotuubef")

if not REPO_DIR.exists():
    print("📥 Cloning repository...")
    run(f"git clone {shlex.quote(REPO_URL)} {shlex.quote(str(REPO_DIR))}")

os.chdir(REPO_DIR)
run("git pull --ff-only || true", allow_fail=True)

print("🐍 Installing Python packages...")
run("python3 -m pip install -q --upgrade pip setuptools wheel")
run("python3 -m pip install -q -r requirements.txt")

# Optional TTS library
qwen_cmd = "python3 -m pip install -q git+https://github.com/QwenLM/Qwen3-TTS.git"
subprocess.run(qwen_cmd, shell=True, text=True)

print("🔨 Building Web Studio frontend assets...")
gui_dir = REPO_DIR / "gui"
if (gui_dir / "package.json").exists():
    subprocess.run(["npm", "install"], cwd=gui_dir, check=False)
    subprocess.run(["npm", "run", "build"], cwd=gui_dir, check=False)

print("✅ Setup complete!")

In [ ]:
# Cell 2: Mount Google Drive & Link Persistence
import os
import shutil
from pathlib import Path
from google.colab import drive

print("💾 Mounting Google Drive...")
drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path("/content/yotuubef")
os.chdir(REPO_DIR)

DRIVE_ROOT = Path("/content/drive/MyDrive/yotuubef")
PERSIST_ROOT = DRIVE_ROOT / "persistent"

important_dirs = {
    "backgrounds": PERSIST_ROOT / "backgrounds",
    "findings": PERSIST_ROOT / "findings",
    "processed": PERSIST_ROOT / "processed",
    "results": PERSIST_ROOT / "results",
    "logs": PERSIST_ROOT / "logs",
    "db": PERSIST_ROOT / "databases",
    "tokens": PERSIST_ROOT / "tokens",
}

for p in [DRIVE_ROOT, PERSIST_ROOT, *important_dirs.values()]:
    p.mkdir(parents=True, exist_ok=True)

def relink_dir(local_path: Path, drive_path: Path) -> None:
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.is_symlink():
        if local_path.resolve(strict=False) == drive_path.resolve(strict=False):
            return
        local_path.unlink()
    elif local_path.exists():
        if not local_path.is_dir():
            return
        shutil.rmtree(local_path)
    local_path.symlink_to(drive_path, target_is_directory=True)

link_map = {
    REPO_DIR / "findings": important_dirs["findings"],
    REPO_DIR / "processed": important_dirs["processed"],
    REPO_DIR / "logs": important_dirs["logs"],
    REPO_DIR / "data" / "results": important_dirs["results"],
    REPO_DIR / "data" / "logs": important_dirs["logs"],
    REPO_DIR / "data" / "databases": important_dirs["db"],
}

for local_dir, drive_dir in link_map.items():
    relink_dir(local_dir, drive_dir)

print("✅ Drive mounted and persistent storage linked.")
print(f"📁 Place your background .mp4 videos in: {important_dirs['backgrounds']}")

In [ ]:
# Cell 3: Load Secrets & Run Preflight Diagnostics
import json
import os
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")
BACKGROUND_FOLDER_PATH = Path("/content/drive/MyDrive/yotuubef/persistent/backgrounds")
TOKEN_FILE_PATH = Path("/content/drive/MyDrive/yotuubef/persistent/tokens/youtube_token.json")

def get_secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value if value is not None else default
    except Exception:
        return os.getenv(name, default)

secret_map = {
    "REDDIT_CLIENT_ID": "REDDIT_CLIENT_ID",
    "REDDIT_CLIENT_SECRET": "REDDIT_CLIENT_SECRET",
    "NVIDIA_NIM_API_KEY": "NVIDIA_NIM_API_KEY",
    "HACKCLUB_SEARCH_API_KEY": "HACKCLUB_SEARCH_API_KEY",
    "YOUTUBE_TOKEN_JSON": "YOUTUBE_TOKEN_JSON",
}

for secret_name, env_name in secret_map.items():
    val = get_secret(secret_name, "")
    if val:
        os.environ[env_name] = val

os.environ["BACKGROUND_FOLDER"] = str(BACKGROUND_FOLDER_PATH)
os.environ["YOUTUBE_TOKEN_FILE"] = str(TOKEN_FILE_PATH)

print("🔍 Environment Configuration Check:")
for key in ["REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET", "NVIDIA_NIM_API_KEY"]:
    status = "✅ Present" if os.environ.get(key) else "⚠️ Missing (can configure in Web GUI Settings)"
    print(f"  - {key}: {status}")

bg_videos = list(BACKGROUND_FOLDER_PATH.glob("*.mp4"))
print(f"  - Background Videos: {len(bg_videos)} .mp4 file(s) found in Drive")

In [ ]:
# Cell 4: Launch Yotuubef Web Studio GUI
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from start_studio import run_colab_studio

print("🚀 Launching Yotuubef Studio Web Application...")
studio_proc = run_colab_studio(port=8420)